# 02 · KDL parse of the full corpus (Arm B's text)

Serves `KDLAI/KDL-Frontier-Parser-nano` with vLLM in this runtime and parses **every page of every PDF** in
`0.1. BENCHMARK`. Client and server share the runtime, so there is no ngrok.

**Input:** `kdl_bundle.zip` from `python research/experiments/prep_ondemand_bundle.py --target kdl`, put in your Drive folder once.
**Output:** in `<Drive folder>/<fingerprint>/kdl/`: `kdl_result.zip` and `kdl_raw_outputs.zip` for the local scripts, and `PASTE_ME.md`
to paste into the chat. The export cell also prints `PASTE_ME.md`.

| | |
|---|---|
| GPU | **L4 24 GB** or A100. Needs 20 GiB and bf16, so **not a T4**. |
| Time | `kdl_pdf_inspector` (default) about 1.3 s/page, roughly **2 h**. Plain `kdl` about 2.4 s/page, roughly **3.4 h**. Expect two sessions. |
| Checkpoint | every document is on local disk as soon as it finishes. Every 5 documents the text checkpoint, and each finished document's raw output, are copied to Drive. |
| Resume | rerun the notebook from the top (the server restarts, about 5 min). The config cell restores the newest checkpoint from Drive by itself. |

**Which parser.** `PROVIDER = "kdl_pdf_inspector"` is the "parse pdf inspector + KDL" row of your results table: native-text
regions are read by pdf-inspector and only the rest goes to KDL. `"kdl"` sends everything to KDL. The choice is recorded in the output.

**What KDL does not tell you** (this notebook handles each): one failing page fails its **whole document**, so failed
documents are retried once and then listed. A page KDL returns nothing for is silently dropped by the provider, so here it is
written as `kdl_status: "empty"`. `max_pages` defaults to 400 and would reject the 892-page industrial manual, so it is raised.

In [ ]:
!nvidia-smi

## 1 · Connect Drive, get the bundle, verify it

Drive is where everything durable goes: checkpoints, raw outputs, the result and `PASTE_ME.md`. The parse itself writes to the
local disk and copies to Drive at each checkpoint, because Drive is a network mount that is slow for many small writes and can drop.
Put `kdl_bundle.zip` in the Drive folder once and every later session reads it from there. If it is not there, you are asked to upload it.

In [ ]:
from google.colab import files
import hashlib, json, os, shutil, time, zipfile
from pathlib import Path

USE_DRIVE    = True
DRIVE_FOLDER = "FOR-COLAB/Colab-for-Axiom-DE-RnD"    # path under My Drive (colab/README.md, "Drive setup")
BUNDLE_NAME  = "kdl_bundle.zip"

REPO = Path("/content/AXIOM_DE-RD"); REPO.mkdir(exist_ok=True)
BENCH = REPO / "0.1. BENCHMARK"

DRIVE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    if not DRIVE.is_dir():
        raise SystemExit(f"no folder {DRIVE_FOLDER!r} in My Drive. It has: {sorted(os.listdir('/content/drive/MyDrive'))}. "
                         "Set DRIVE_FOLDER to its path under My Drive. If the folders you expect are not listed at all, Colab mounted "
                         "a different Google account: run drive.flush_and_unmount(), then rerun this cell and pick the account that owns the folder.")

bundle_zip = Path("/content") / BUNDLE_NAME
if DRIVE and (DRIVE / BUNDLE_NAME).exists():
    shutil.copy(DRIVE / BUNDLE_NAME, bundle_zip)       # unzip locally: 103 PDFs written onto Drive would be minutes of churn
    print("bundle read from Drive")
else:
    uploaded = files.upload()                          # choose kdl_bundle.zip
    name = next(n for n in uploaded if n.endswith(".zip")); del uploaded
    if Path(name).resolve() != bundle_zip.resolve():
        shutil.move(name, bundle_zip)
    if DRIVE:
        shutil.copy(bundle_zip, DRIVE / BUNDLE_NAME); print("bundle saved to Drive for the next session")
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall(REPO)

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

manifest = json.loads((REPO / "BUNDLE_MANIFEST.json").read_text())
assert manifest["target"] == "kdl", f"this notebook needs a kdl bundle, got {manifest['target']!r}"
bad = [rel for rel, digest in manifest["files"].items() if _sha256(REPO / rel) != digest]
assert not bad, f"{len(bad)} files are corrupt or missing (truncated upload?), e.g. {bad[:3]}"

h = hashlib.sha256()
for name in ("documents.jsonl", "queries.jsonl", "qrels.jsonl"):
    h.update(name.encode()); h.update((BENCH / name).read_bytes())
FINGERPRINT = h.hexdigest()[:12]
assert FINGERPRINT == manifest["bundle_fingerprint"], "manifests do not match the bundle fingerprint"

documents = [json.loads(l) for l in (BENCH / "documents.jsonl").read_text().splitlines() if l.strip()]
print(f"bundle {FINGERPRINT}{'  (SMOKE)' if manifest['smoke'] else ''}: {len(documents)} documents, "
      f"{manifest['n_pages']} pages, {len(manifest['files'])} files verified")
print("largest documents:", sorted(((d["metadata"]["page_count"], d["metadata"].get("source_doc_id", d["doc_id"])) for d in documents), reverse=True)[:3])

# One folder per bundle and notebook, so a smoke run and the full run never share checkpoints.
RUN = (DRIVE / FINGERPRINT / "kdl") if DRIVE else Path("/content/kdl_run")
RUN.mkdir(parents=True, exist_ok=True)

def to_drive(path, folder=None):
    """Copy a file into the run folder. A Drive error is printed, never raised: the local copy is still there."""
    dest = (folder or RUN) / Path(path).name
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        part = dest.with_name(dest.name + ".part")
        shutil.copy(path, part); part.replace(dest)
        return dest
    except OSError as e:
        part.unlink(missing_ok=True)
        print(f"  could not copy {Path(path).name} to {dest.parent}: {e}. The local copy is kept.")

if DRIVE:                                  # prove a write + rename + read works now, not two hours in
    probe = Path("/content/drive_probe.txt"); probe.write_text(FINGERPRINT)
    copied = to_drive(probe)
    if not copied or copied.read_text() != FINGERPRINT:
        raise SystemExit("writing to Drive does not work (see the message above). Fix it, or set USE_DRIVE = False to use browser downloads.")
    copied.unlink()
print("run folder:", RUN)

## 2 · Install

Same order as `KDL_serving_de.ipynb`, which worked: vLLM first, and the repo only once the server is up (cell 4).
If pip reports a conflict, **Runtime > Restart session** and continue from cell 3 (files under `/content` survive a restart; Drive has to be mounted again, which cell 3 does).

In [ ]:
%pip install -q "vllm==0.19.0"
# The repo is installed in cell 4, AFTER the server is running. The proven KDL_serving_de.ipynb did it in that order:
# a running server is unaffected by later pip changes, while installing the repo first can move numpy/pydantic/fastapi
# under vLLM before it has even started.

## 3 · GPU profile

In [ ]:
import os, torch

assert torch.cuda.is_available(), "no CUDA"
assert torch.cuda.is_bf16_supported(), "this GPU has no bf16: use an L4 or A100, not a T4"
gpu = torch.cuda.get_device_properties(0)
gpu_name, vram_gib, cpu_count = gpu.name, gpu.total_memory / 1024**3, os.cpu_count() or 1
if vram_gib < 20:
    raise RuntimeError(f"{gpu_name} has {vram_gib:.0f} GiB; KDL needs about 20 GiB: use an L4 or A100")

# Per-GPU tuning, unchanged from KDL_serving_de.ipynb.
if "H100" in gpu_name:
    VLLM_DTYPE, VLLM_MAX_NUM_SEQS, VLLM_MAX_BATCHED_TOKENS, VLLM_GPU_MEMORY_UTILIZATION = "bfloat16", "256", "65536", "0.90"
    KDL_MAX_WORKERS, KDL_BBOX_MAX_WORKERS, KDL_RENDER_PROCESSES = 64, 256, min(32, cpu_count)
elif "A100" in gpu_name and vram_gib >= 70:
    VLLM_DTYPE, VLLM_MAX_NUM_SEQS, VLLM_MAX_BATCHED_TOKENS, VLLM_GPU_MEMORY_UTILIZATION = "bfloat16", "128", "32768", "0.90"
    KDL_MAX_WORKERS, KDL_BBOX_MAX_WORKERS, KDL_RENDER_PROCESSES = 48, 128, min(24, cpu_count)
elif "A100" in gpu_name:
    VLLM_DTYPE, VLLM_MAX_NUM_SEQS, VLLM_MAX_BATCHED_TOKENS, VLLM_GPU_MEMORY_UTILIZATION = "bfloat16", "64", "16384", "0.95"
    KDL_MAX_WORKERS, KDL_BBOX_MAX_WORKERS, KDL_RENDER_PROCESSES = 32, 64, min(16, cpu_count)
elif "L4" in gpu_name:
    VLLM_DTYPE, VLLM_MAX_NUM_SEQS, VLLM_MAX_BATCHED_TOKENS, VLLM_GPU_MEMORY_UTILIZATION = "bfloat16", "32", "8192", "0.92"
    KDL_MAX_WORKERS, KDL_BBOX_MAX_WORKERS, KDL_RENDER_PROCESSES = 16, 32, min(8, cpu_count)
else:
    VLLM_DTYPE, VLLM_MAX_NUM_SEQS, VLLM_MAX_BATCHED_TOKENS, VLLM_GPU_MEMORY_UTILIZATION = "auto", "8", "4096", "0.90"
    KDL_MAX_WORKERS, KDL_BBOX_MAX_WORKERS, KDL_RENDER_PROCESSES = 8, 16, min(4, cpu_count)

print(f"{gpu_name} {vram_gib:.0f} GiB, {cpu_count} CPUs | vLLM seqs {VLLM_MAX_NUM_SEQS}, batched tokens {VLLM_MAX_BATCHED_TOKENS}, "
      f"util {VLLM_GPU_MEMORY_UTILIZATION} | KDL workers {KDL_MAX_WORKERS}, bbox {KDL_BBOX_MAX_WORKERS}, render {KDL_RENDER_PROCESSES}")

## 4 · Start the vLLM server and wait for it

In [ ]:
import subprocess, requests
from importlib.metadata import version as _version

assert _version("vllm") == "0.19.0", f"vllm {_version('vllm')} is installed, not 0.19.0: Runtime > Restart session and rerun the install cell"

VLLM_LOG = Path("/content/vllm-kdl.log")
VLLM_ARGS = ["vllm", "serve", "KDLAI/KDL-Frontier-Parser-nano", "--host", "0.0.0.0", "--port", "8000",
             "--served-model-name", "kdl-frontier-parser-nano", "--dtype", VLLM_DTYPE, "--max-model-len", "8192",
             "--max-num-seqs", VLLM_MAX_NUM_SEQS, "--max-num-batched-tokens", VLLM_MAX_BATCHED_TOKENS,
             "--gpu-memory-utilization", VLLM_GPU_MEMORY_UTILIZATION, "--limit-mm-per-prompt", '{"image":1}',
             "--trust-remote-code", "--enable-chunked-prefill", "--enable-prefix-caching", "--generation-config", "vllm"]

def server_up():
    try:
        return requests.get("http://127.0.0.1:8000/v1/models", timeout=5).status_code == 200
    except requests.RequestException:
        return False

if server_up():
    print("a vLLM server is already answering on :8000, reusing it")
    vllm_proc = None
else:
    vllm_proc = subprocess.Popen(VLLM_ARGS, stdout=open(VLLM_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0, deadline = time.time(), time.time() + 30 * 60
    while not server_up():
        if vllm_proc.poll() is not None:
            print(VLLM_LOG.read_text()[-4000:])
            raise RuntimeError(f"vLLM exited with code {vllm_proc.returncode} while starting; log tail above")
        if time.time() > deadline:
            print(VLLM_LOG.read_text()[-4000:])
            raise RuntimeError("vLLM was not ready after 30 min")
        time.sleep(10)
    print(f"vLLM ready after {time.time() - t0:.0f}s")

VLLM_API_BASE = "http://127.0.0.1:8000/v1"
MODEL_NAME = requests.get(f"{VLLM_API_BASE}/models", timeout=30).json()["data"][0]["id"]
paths = requests.get("http://127.0.0.1:8000/openapi.json", timeout=30).json().get("paths", {})
print("served model:", MODEL_NAME, "| batch endpoint /v1/chat/completions/batch:", "/v1/chat/completions/batch" in paths)

# Only now install the repo (see cell 2). Then prove both halves still work, so a conflict shows up here and not two hours in.
%cd /content/AXIOM_DE-RD
%pip install -q -e ".[pdf-inspector]"
import sys
sys.path.insert(0, str(REPO))
from src.ingestion.parsing.kdl_pdf_inspector import KdlPdfInspectorProvider   # imports the KDL client and pdf-inspector
from src.ingestion.parsing.kdl import KDLProvider
assert server_up(), "the repo install broke the running vLLM server: restart the runtime and rerun from cell 2"
print("repo installed; vLLM still answering; KDL client imports")

## 5 · Config

In [ ]:
import json, zipfile
from importlib.metadata import version

PROVIDER           = "kdl_pdf_inspector"   # or "kdl". See the note at the top.
SCHEDULER          = "parsebench_document"  # "global_two_phase" parallelises pages inside one document; needs the batch endpoint above
REQUEST_BATCH_SIZE = 1                      # >1 needs /v1/chat/completions/batch
RESUME             = True                   # continue from the newest checkpoint in the run folder, if there is one
CKPT_EVERY_DOCS    = 5                      # copy a checkpoint to Drive this often, and always when the parse cell ends
LIMIT_DOCS         = 0                      # >0: parse only the first N documents (a quick trial)

OUT = Path("/content/kdl_out"); OUT.mkdir(exist_ok=True)
if RESUME:
    found = sorted(RUN.glob("kdl_ckpt_*.zip"))
    if not found and not DRIVE:
        up = files.upload()
        found = [Path(sorted(n for n in up if n.endswith(".zip"))[-1])]
    if found:
        with zipfile.ZipFile(found[-1]) as zf:
            ck = json.loads(zf.read("kdl_done.json"))
            assert ck["bundle"] == FINGERPRINT, "that checkpoint is from a different bundle"
            zf.extractall(OUT)
        raw_zips = sorted((RUN / "raw").glob("*.zip"))
        for z in raw_zips:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(OUT / "raw" / z.stem)
        print(f"resumed from {found[-1].name}: {len(ck['documents'])} documents already parsed, {len(raw_zips)} raw outputs restored")
    else:
        print(f"no checkpoint in {RUN}: starting from the first document")

ENV = {k: version(k) for k in ("vllm", "torch", "transformers", "pymupdf", "httpx")}
print("provider", PROVIDER, "| scheduler", SCHEDULER, "|", ENV)

## 6 · Parse

Runs `research/experiments/kdl_parse_corpus.py` as a subprocess and streams its log. Safe to interrupt: every finished
document is on disk, and rerunning this cell skips them and retries any that failed.

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

sys.path.insert(0, str(REPO))
from research.experiments.kdl_parse_corpus import safe_id    # the runner's name for a document's raw folder

RAW_ZIPS = Path("/content/kdl_raw_zips"); RAW_ZIPS.mkdir(exist_ok=True)
raw_synced = {z.stem for z in (RUN / "raw").glob("*.zip")}

def make_checkpoint():
    """Zip the resume files, named by the total documents done so the newest sorts last, and copy it and every
    finished document's raw output to the run folder."""
    if not (OUT / "kdl_done.json").exists():
        return None
    done_ids = json.loads((OUT / "kdl_done.json").read_text())["documents"]
    path = Path(f"/content/kdl_ckpt_{len(done_ids):04d}.zip")
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in ("kdl_pages.jsonl", "kdl_doc_meta.jsonl", "kdl_done.json", "kdl_failed.jsonl", "timings.jsonl"):
            if (OUT / name).exists():
                zf.write(OUT / name, name)
    for doc_id in done_ids:                  # raw outputs first: a checkpoint must never list a document whose raw output is not there
        name, folder = safe_id(doc_id), OUT / "raw" / safe_id(doc_id)
        if name not in raw_synced and folder.is_dir():
            if to_drive(shutil.make_archive(str(RAW_ZIPS / name), "zip", folder), RUN / "raw"):
                raw_synced.add(name)
    if to_drive(path):
        for old in sorted(RUN.glob("kdl_ckpt_*.zip"))[:-2]:     # keep the newest two
            old.unlink(missing_ok=True)
    return path

def stream(cmd, env, on_doc_done):
    proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
        if line.startswith("DOC ") and " ok " in line:
            on_doc_done()
    return proc.wait()

CMD = [sys.executable, "research/experiments/kdl_parse_corpus.py", "--bench", str(BENCH), "--out", str(OUT),
       "--provider", PROVIDER, "--scheduler", SCHEDULER, "--max-workers", str(KDL_MAX_WORKERS),
       "--render-processes", str(KDL_RENDER_PROCESSES), "--bbox-max-workers", str(KDL_BBOX_MAX_WORKERS),
       "--request-batch-size", str(REQUEST_BATCH_SIZE), "--max-model-sequences", VLLM_MAX_NUM_SEQS,
       "--endpoint", VLLM_API_BASE, "--model", MODEL_NAME] + (["--limit-docs", str(LIMIT_DOCS)] if LIMIT_DOCS else [])
ENVIRON = {**os.environ, "PYTHONPATH": str(REPO), "VLLM_API_BASE": VLLM_API_BASE, "VLLM_MODEL_NAME": MODEL_NAME,
           "KDL_GPU_NAME": gpu_name}

finished_here = 0
def on_doc_done():
    global finished_here
    finished_here += 1
    if finished_here % CKPT_EVERY_DOCS == 0:
        ck = make_checkpoint()
        if not DRIVE and finished_here % 20 == 0:
            files.download(str(ck))                   # no Drive: the browser is the only copy that survives a disconnect

t0 = time.time()
try:
    return_code = stream(CMD, ENVIRON, on_doc_done)
finally:
    make_checkpoint()                                 # also on an interrupt: nothing finished is lost
session_seconds = time.time() - t0
print(f"\nexit code {return_code} after {session_seconds / 60:.1f} min" + ("  <- documents failed, see kdl_failed.jsonl; rerun this cell to retry" if return_code == 2 else ""))

## 7 · Check, export, and the text to paste back

Writes `kdl_result.zip`, `kdl_raw_outputs.zip` and `PASTE_ME.md` to the Drive run folder, and prints `PASTE_ME.md` so you can copy it
straight from this page.

In [ ]:
import json, shutil, collections
import numpy as np
from pathlib import Path

done = set(json.loads((OUT / "kdl_done.json").read_text())["documents"])
want = {d["doc_id"] for d in documents[:LIMIT_DOCS] if LIMIT_DOCS} or {d["doc_id"] for d in documents}
missing = sorted(want - done)
pages = [json.loads(l) for l in (OUT / "kdl_pages.jsonl").read_text().splitlines() if l.strip()]
expected = sum(d["metadata"]["page_count"] for d in documents if d["doc_id"] in want)
print(f"{len(done & want)}/{len(want)} documents done | {len(pages)}/{expected} page rows")
by = collections.defaultdict(lambda: [0, 0])
for r in pages:
    by[r["source"]][0] += r["kdl_status"] == "ok"; by[r["source"]][1] += 1
for s, (ok_, n) in sorted(by.items()):
    print(f"  {s:20s} {ok_}/{n} pages have text ({100 * ok_ / max(n, 1):.1f}%)")
assert not missing, f"{len(missing)} documents not parsed, e.g. {missing[:3]}. Rerun the cell above."
assert len(pages) == expected and len({r['page_id'] for r in pages}) == len(pages)
no_raw = sorted(d for d in done & want if not (OUT / "raw" / safe_id(d) / "result.json").exists())
assert not no_raw, (f"{len(no_raw)} parsed documents have no raw output (lost in a disconnect?), e.g. {no_raw[:3]}. "
                    "Remove them from kdl_out/kdl_done.json and rerun the parse cell to parse them again.")

RES = Path("/content/kdl_result"); RES.mkdir(exist_ok=True)
for name in ("kdl_pages.jsonl", "kdl_doc_meta.jsonl", "kdl_done.json", "kdl_failed.jsonl", "timings.jsonl"):
    if (OUT / name).exists():
        shutil.copy(OUT / name, RES / name)
meta = {"provider": PROVIDER, "scheduler": SCHEDULER, "request_batch_size": REQUEST_BATCH_SIZE, "model": MODEL_NAME,
        "served_model": "KDLAI/KDL-Frontier-Parser-nano", "dpi": 144, "bundle_fingerprint": FINGERPRINT,
        "smoke": manifest["smoke"], "gpu": gpu_name, "env": ENV, "vllm_args": VLLM_ARGS,
        "kdl_workers": {"max_workers": KDL_MAX_WORKERS, "render_processes": KDL_RENDER_PROCESSES, "bbox_max_workers": KDL_BBOX_MAX_WORKERS},
        "n_documents": len(want), "n_pages": len(pages), "pages_with_text": sum(v[0] for v in by.values())}
json.dump(meta, open(RES / "kdl_meta.json", "w"), indent=1)

def timing_summary(rows):
    """Per stage: how many records and units, total seconds, and the per-unit spread (so a stall is visible, not averaged away)."""
    out = {}
    for stage in dict.fromkeys(r["stage"] for r in rows):
        rs = [r for r in rows if r["stage"] == stage]
        per = np.array([r["seconds"] / max(r["n"], 1) for r in rs])
        s = {"unit": rs[0]["unit"], "records": len(rs), "units": int(sum(r["n"] for r in rs)),
             "total_seconds": round(float(sum(r["seconds"] for r in rs)), 2), "per_unit_mean": round(float(per.mean()), 4),
             "per_unit_p50": round(float(np.percentile(per, 50)), 4), "per_unit_p95": round(float(np.percentile(per, 95)), 4)}
        for k in sorted({k for r in rs for k in r if k.endswith("_seconds")}):
            s[f"mean_{k}"] = round(float(np.mean([r[k] for r in rs if k in r])), 4)
        out[stage] = s
    return out

rows = [json.loads(l) for l in (OUT / "timings.jsonl").read_text().splitlines() if l.strip()] if (OUT / "timings.jsonl").exists() else []
summary = timing_summary(rows)
json.dump(summary, open(RES / "timings_summary.json", "w"), indent=1)
failed = [json.loads(l) for l in (OUT / "kdl_failed.jsonl").read_text().splitlines() if l.strip()] if (OUT / "kdl_failed.jsonl").exists() else []
coverage = {s: {"pages_with_text": ok_, "pages": n} for s, (ok_, n) in sorted(by.items())}
paste = "\n".join([
    f"# kdl result, bundle {FINGERPRINT}{' (SMOKE)' if manifest['smoke'] else ''}",
    f"written {time.strftime('%Y-%m-%d %H:%M')} | {len(done & want)}/{len(want)} documents | {len(pages)} pages",
    "", "## kdl_meta.json", "```json", json.dumps(meta, indent=1), "```",
    "", "## pages with text, per source", "```json", json.dumps(coverage, indent=1), "```",
    "", "## documents that failed at least once (kdl_failed.jsonl)", "```json", json.dumps(failed, indent=1)[:3000], "```",
    "", "## timings_summary.json", "```json", json.dumps(summary, indent=1), "```", ""])
(RES / "PASTE_ME.md").write_text(paste)

shutil.make_archive("/content/kdl_result", "zip", RES)
shutil.make_archive("/content/kdl_raw_outputs", "zip", OUT, "raw")
print("kdl_result.zip", round(Path("/content/kdl_result.zip").stat().st_size / 1e6, 1), "MB |",
      "kdl_raw_outputs.zip", round(Path("/content/kdl_raw_outputs.zip").stat().st_size / 1e6, 1), "MB")
on_drive = {f.name: to_drive(f) if DRIVE else None for f in (Path("/content/kdl_result.zip"), Path("/content/kdl_raw_outputs.zip"), RES / "PASTE_ME.md")}
print("\n".join(f"  on Drive: {p}" for p in on_drive.values() if p))

In [ ]:
for name in ("kdl_result.zip", "kdl_raw_outputs.zip"):      # no Drive, or the Drive copy failed: the browser is the only way out
    if not on_drive[name]:
        files.download(f"/content/{name}")     # layout elements with bounding boxes: what lets us trace a bad chunk to its region on the page
print("=" * 30, "copy everything below this line into the chat", "=" * 30)
print((RES / "PASTE_ME.md").read_text())

## 8 · Stop the server

Free the GPU. If you are about to resume in a new session this is moot, since the runtime is recycled anyway.

In [ ]:
import os, signal
if vllm_proc is not None and vllm_proc.poll() is None:
    os.killpg(os.getpgid(vllm_proc.pid), signal.SIGTERM)
    print("vLLM stopped")